# Food-Sure: Vote Test — ResNet50 and the new pipeline deciding together (3 classes)

**Plan agreed by the team.** Both models answer "pure, adulterant A or adulterant B" and the app averages their
opinions. The level (shown as a low / medium / high band) then comes from ResNet50.
This notebook checks that exact setup on the held-out test photos before it goes into the backend, and packs
the files the backend machine needs into `Food Sure/fair_comparison/vote_test/deploy_bundle.zip`.

Read-only with respect to existing work: only writes to `Food Sure/fair_comparison/vote_test/`.

## 1 — Setup

In [ ]:
import os, sys, subprocess
LOCAL = os.environ.get('FS_LOCAL') == '1'      # only used for an offline dry run with dummy data
if not LOCAL:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'catboost', 'lightgbm'], check=True)

In [ ]:
import re, json, warnings
import numpy as np, pandas as pd
from scipy.stats import skew, kurtosis, spearmanr
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import RidgeCV
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, VotingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
import catboost as cb
import lightgbm as lgb
import matplotlib
if LOCAL:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR       = os.environ.get('FS_BASE', '/content/drive/MyDrive/Food Sure')
SPLIT_DIR      = os.path.join(BASE_DIR, 'dataset_split')
WEB_MODELS_DIR = os.path.join(BASE_DIR, 'web_models')
FEAT_DIR       = os.path.join(BASE_DIR, 'fair_comparison')                 # cached handcrafted features live here
OUT_DIR        = os.path.join(FEAT_DIR, 'concentration_test')              # new folder, only outputs go here
os.makedirs(OUT_DIR, exist_ok=True)

SPICES = ['black_pepper_powder', 'red_chilli_powder', 'turmeric']
SEED = 42
CROP_SIZE, CROPS_PER_IMAGE = 256, 4
VALID_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')
HIDDEN_LEVEL = 10.0

for s in SPICES:
    for part in ['train', 'val', 'test']:
        assert os.path.isdir(os.path.join(SPLIT_DIR, s, part)), f'missing {s}/{part}'
print('Paths OK')

## 2 — Data helpers and the new pipeline's code
Feature extractor, crops, classifier and regressor settings are copied unchanged from `Untitled5.ipynb`.

In [ ]:
def to_type(label):
    label = str(label).lower()
    return 'pure' if label == 'pure' else re.sub(r'^\d+_pct_', '', label)

def pct_of(label):
    m = re.match(r'^(\d+)_pct_', str(label).lower())
    return float(m.group(1)) if m else 0.0

def list_split(spice, part):
    root = os.path.join(SPLIT_DIR, spice, part)
    items = []
    for cls in sorted(os.listdir(root)):
        d = os.path.join(root, cls)
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            if f.lower().endswith(VALID_EXTS):
                items.append((os.path.join(d, f), cls.lower()))
    return items


def extract_80plus_features(img_bgr, use_gray_world=False):
    import cv2
    from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
    if use_gray_world:
        b, g, r = cv2.split(img_bgr.astype(np.float32))
        gray_avg = (np.mean(b) + np.mean(g) + np.mean(r)) / 3.0
        b = np.clip(b * (gray_avg / (np.mean(b) + 1e-5)), 0, 255)
        g = np.clip(g * (gray_avg / (np.mean(g) + 1e-5)), 0, 255)
        r = np.clip(r * (gray_avg / (np.mean(r) + 1e-5)), 0, 255)
        img_bgr = cv2.merge([b, g, r]).astype(np.uint8)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    img_ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    feats = []
    for space in [img_rgb, img_hsv, img_lab, img_ycrcb]:
        for ch in range(3):
            plane = space[:, :, ch].astype(np.float32).ravel()
            mean_v, std_v = np.mean(plane), np.std(plane)
            sk_v = float(skew(plane)) if std_v > 1e-3 else 0.0
            kt_v = float(kurtosis(plane)) if std_v > 1e-3 else 0.0
            feats.extend([mean_v, std_v, sk_v, kt_v])
    h_hist, _ = np.histogram(img_hsv[:, :, 0], bins=16, range=(0, 180), density=True)
    s_hist, _ = np.histogram(img_hsv[:, :, 1], bins=16, range=(0, 256), density=True)
    l_hist, _ = np.histogram(img_lab[:, :, 0], bins=16, range=(0, 256), density=True)
    a_hist, _ = np.histogram(img_lab[:, :, 1], bins=16, range=(0, 256), density=True)
    b_hist, _ = np.histogram(img_lab[:, :, 2], bins=16, range=(0, 256), density=True)
    feats.extend(np.hstack([h_hist, s_hist, l_hist, a_hist, b_hist]))
    lab_l = img_lab[:, :, 0].astype(np.float32)
    lab_a = img_lab[:, :, 1].astype(np.float32)
    lab_b = img_lab[:, :, 2].astype(np.float32)
    hsv_s = img_hsv[:, :, 1].astype(np.float32)
    cai = np.mean(lab_b) / (np.mean(lab_l) + 1e-5)
    rpi = np.mean(lab_a) / (np.mean(hsv_s) + 1e-5)
    fleck_ratio = np.sum(img_gray > 140) / (img_gray.size + 1e-5)
    lap_var = float(np.var(cv2.Laplacian(img_gray, cv2.CV_64F)))
    feats.extend([cai, rpi, fleck_ratio, lap_var])
    glcm = graycomatrix(img_gray, distances=[1, 2], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)
    glcm_iso = np.mean(glcm, axis=3, keepdims=True)
    for prop in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']:
        feats.extend(graycoprops(glcm_iso, prop).ravel())
    lbp = local_binary_pattern(img_gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10), density=True)
    feats.extend(lbp_hist)
    return np.array(feats, dtype=np.float32)


def image_crops(img):
    import cv2
    h, w = img.shape[:2]
    if h >= CROP_SIZE and w >= CROP_SIZE:
        coords = [((h - CROP_SIZE) // 2, (w - CROP_SIZE) // 2), (0, 0),
                  (h - CROP_SIZE, w - CROP_SIZE), (0, w - CROP_SIZE)]
        return [img[t:t + CROP_SIZE, l:l + CROP_SIZE] for t, l in coords[:CROPS_PER_IMAGE]]
    return [cv2.resize(img, (CROP_SIZE, CROP_SIZE))]


def featurize(spice, part_name, items, use_gray_world):
    # Features per crop; groups[i] = index of the source photo in `items`. Cached (same files as the fair comparison).
    cache = os.path.join(FEAT_DIR, f'feats_{spice}_{part_name}_gw{int(use_gray_world)}.npz')
    if os.path.exists(cache):
        d = np.load(cache, allow_pickle=True)
        return d['X'], d['y'], d['groups']
    import cv2
    from tqdm import tqdm
    X, y, groups = [], [], []
    for gid, (p, label) in enumerate(tqdm(items, desc=f'{spice}/{part_name}')):
        img = cv2.imread(p)
        if img is None:
            print('unreadable, skipped:', p)
            continue
        for patch in image_crops(img):
            X.append(extract_80plus_features(patch, use_gray_world=use_gray_world))
            y.append(label)
            groups.append(gid)
    X, y, groups = np.array(X), np.array(y), np.array(groups)
    np.savez(cache, X=X, y=y, groups=groups)
    return X, y, groups


def load_spice(spice):
    # All photos of one spice. Photo id = position in train+val+test listing; ids >= n_tv are the test photos.
    gw = (spice == 'red_chilli_powder')
    tv, te = list_split(spice, 'train') + list_split(spice, 'val'), list_split(spice, 'test')
    Xa, ya, ga = featurize(spice, 'trainval', tv, gw)
    Xb, yb, gb = featurize(spice, 'test', te, gw)
    y7 = np.concatenate([ya, yb]).astype(str)
    return {'spice': spice, 'items': tv + te, 'n_tv': len(tv),
            'X': np.vstack([Xa, Xb]).astype(np.float32),
            'g': np.concatenate([ga, gb + len(tv)]).astype(int),
            'y3': np.array([to_type(l) for l in y7]),
            'pct': np.array([pct_of(l) for l in y7])}


def make_clf(spice):
    if spice == 'red_chilli_powder':  # Pipeline 2
        m1 = cb.CatBoostClassifier(iterations=550, learning_rate=0.035, depth=5, auto_class_weights='Balanced', verbose=0, random_seed=42)
        m2 = lgb.LGBMClassifier(n_estimators=350, learning_rate=0.035, max_depth=5, class_weight='balanced', random_state=42, verbose=-1)
        m3 = ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=42)
        return VotingClassifier(estimators=[('cb', m1), ('lgb', m2), ('et', m3)], voting='soft')
    return cb.CatBoostClassifier(iterations=600, learning_rate=0.04, depth=5, auto_class_weights='Balanced', verbose=0, random_seed=42)


def classifier_fit_predict(spice, X_tr, y_tr, X_te, groups_te):
    # Stage 1 of the new pipeline. Returns (photo ids, class names, per-photo probabilities averaged over crops)
    scaler = RobustScaler().fit(X_tr)
    X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)
    selector = SelectKBest(score_func=f_classif, k=min(65, X_tr.shape[1])).fit(X_tr_s, y_tr)
    clf = make_clf(spice)
    clf.fit(selector.transform(X_tr_s), y_tr)
    probs = clf.predict_proba(selector.transform(X_te_s))
    ids = np.unique(groups_te)
    return ids, np.array(clf.classes_).astype(str), np.stack([probs[groups_te == g].mean(axis=0) for g in ids])


def fit_extratrees(X_tr, pct_tr):
    # Stage 2 of the new pipeline, settings unchanged: RobustScaler -> SelectKBest(f_regression, 45) -> ExtraTrees
    scaler = RobustScaler().fit(X_tr)
    Xs = scaler.transform(X_tr)
    sel = SelectKBest(score_func=f_regression, k=min(45, X_tr.shape[1])).fit(Xs, pct_tr)
    reg = ExtraTreesRegressor(n_estimators=300, max_depth=9, min_samples_split=4, min_samples_leaf=2,
                              random_state=42, n_jobs=-1).fit(sel.transform(Xs), pct_tr)
    return lambda X: reg.predict(sel.transform(scaler.transform(X)))


def fit_ridge(X_tr, pct_tr):
    # A plain linear regressor, as a second opinion: straight lines can interpolate, trees can only average
    m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 13))).fit(X_tr, pct_tr)
    return m.predict

REGRESSORS = {'ExtraTrees (new pipeline)': fit_extratrees, 'Ridge (linear)': fit_ridge}

def photo_median(values, g, ids):
    return np.array([np.median(values[g == i]) for i in ids])

## 3 — ResNet50: the deployed models, and frozen image features
* **Deployed ResNet50** (`web_models/`): its 7 class probabilities give a percentage two ways — the top class
  (what the app shows today) and a probability-weighted average of 5/10/15.
* **Frozen ResNet50 features**: the standard ImageNet ResNet50 with no fine-tuning, used only as a feature
  extractor (2048 numbers per photo). It has never seen our labels, so it can be used in the hide-the-10% test.
  The deployed models cannot: they were trained with the 10% photos.

In [ ]:
def resnet_predict(spice, items):
    with open(os.path.join(WEB_MODELS_DIR, spice, 'label_map.json')) as f:
        label_map = json.load(f)
    classes = np.array([label_map[str(i)].lower() for i in range(len(label_map))])
    if LOCAL:   # dry run: noisy but informative fake probabilities
        rng = np.random.default_rng(1)
        probs = rng.dirichlet(np.ones(len(classes)), len(items))
        for k, (_, lab) in enumerate(items):
            probs[k, list(classes).index(lab)] += 1.0
        return classes, probs / probs.sum(1, keepdims=True)
    import tensorflow as tf
    model = tf.keras.models.load_model(os.path.join(WEB_MODELS_DIR, spice, 'model.keras'))
    batch = np.stack([tf.keras.utils.img_to_array(tf.keras.utils.load_img(p, target_size=(224, 224)))
                      for p, _ in items])
    probs = model.predict(batch, batch_size=16, verbose=0)
    del model
    tf.keras.backend.clear_session()
    return classes, probs


def collapse_to_type(classes, probs):
    types = np.array([to_type(c) for c in classes])
    tclasses = np.unique(types)
    return tclasses, np.stack([probs[:, types == t].sum(axis=1) for t in tclasses], axis=1)


_backbone = None
def embed_spice(spice, items):
    # One 2048-number vector per photo from the frozen ImageNet ResNet50. Cached to OUT_DIR. NaN row = unreadable.
    global _backbone
    cache = os.path.join(OUT_DIR, f'emb_{spice}.npz')
    if os.path.exists(cache):
        return np.load(cache)['E']
    if LOCAL:
        rng = np.random.default_rng(2)
        E = rng.normal(size=(len(items), 64)).astype(np.float32)
        E[:, 0] += np.array([pct_of(l) for _, l in items]) / 5.0
        return E
    import tensorflow as tf
    from tqdm import tqdm
    if _backbone is None:
        _backbone = tf.keras.applications.ResNet50(include_top=False, weights='imagenet', pooling='avg')
    E = np.full((len(items), 2048), np.nan, dtype=np.float32)
    for start in tqdm(range(0, len(items), 32), desc=f'embedding {spice}'):
        batch, idx = [], []
        for i in range(start, min(start + 32, len(items))):
            try:
                batch.append(tf.keras.utils.img_to_array(tf.keras.utils.load_img(items[i][0], target_size=(224, 224))))
                idx.append(i)
            except Exception as e:
                print('unreadable, skipped:', items[i][0], e)
        if batch:
            x = tf.keras.applications.resnet50.preprocess_input(np.stack(batch))
            E[idx] = _backbone.predict(x, verbose=0)
    np.savez(cache, E=E)
    return E

## 4 — The vote, on the same held-out test photos
* **Pipeline** is trained on the train+val photos with 3 classes, exactly as in `Untitled5.ipynb`.
* **ResNet50** is the deployed model; its 7 probabilities are added up per adulterant to give 3.
* **Vote** = plain average of the two (50/50, fixed in advance, nothing tuned).
* **Level** = once the adulterant is decided, ResNet50's most likely level (5 / 10 / 15) for that adulterant,
  shown in the app as a band: low (under 7.5%), medium (7.5–12.5%), high (above 12.5%).

In [ ]:
import shutil, zipfile
from scipy.stats import binomtest

VOTE_DIR = os.path.join(FEAT_DIR, 'vote_test')
os.makedirs(VOTE_DIR, exist_ok=True)

def mcnemar(correct_a, correct_b):
    b = int(np.sum(correct_a & ~correct_b)); c = int(np.sum(~correct_a & correct_b))
    return (1.0 if b + c == 0 else float(binomtest(b, b + c, 0.5).pvalue)), b, c

def level_from_resnet(types, r_cls7, r_p7):
    # For each photo: given the decided type, ResNet50's most likely class of that type
    r_types = np.array([to_type(c) for c in r_cls7])
    out = []
    for t, p in zip(types, r_p7):
        idx = np.where(r_types == t)[0]
        out.append(r_cls7[idx[np.argmax(p[idx])]])
    return np.array(out)

rows, ref = [], []
bundle = os.path.join('/content' if not LOCAL else VOTE_DIR, 'deploy_bundle')
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(os.path.join(bundle, 'features'), exist_ok=True)

for spice in SPICES:
    D = load_spice(spice)
    te_rows = D['g'] >= D['n_tv']; tr_rows = ~te_rows
    te_ids = np.unique(D['g'][te_rows])
    true7 = np.array([D['items'][i][1] for i in te_ids])
    true3 = np.array([to_type(l) for l in true7])

    ids, p_cls, p_p3 = classifier_fit_predict(spice, D['X'][tr_rows], D['y3'][tr_rows], D['X'][te_rows], D['g'][te_rows])
    assert np.array_equal(ids, te_ids)
    r_cls7, r_p7 = resnet_predict(spice, D['items'][D['n_tv']:])
    r_p7 = r_p7[te_ids - D['n_tv']]
    r_cls3, r_p3 = collapse_to_type(r_cls7, r_p7)
    order = np.array(sorted(r_cls3))
    assert sorted(p_cls) == list(order), (p_cls, order)
    p_p3 = p_p3[:, [list(p_cls).index(c) for c in order]]
    r_p3 = r_p3[:, [list(r_cls3).index(c) for c in order]]
    v_p3 = 0.5 * r_p3 + 0.5 * p_p3

    types = {'ResNet50 alone': order[r_p3.argmax(1)], 'Pipeline alone': order[p_p3.argmax(1)], 'Both voting': order[v_p3.argmax(1)]}
    full = {'ResNet50 alone': r_cls7[r_p7.argmax(1)],
            'Both voting': level_from_resnet(types['Both voting'], r_cls7, r_p7)}
    base3, base7 = types['ResNet50 alone'] == true3, full['ResNet50 alone'] == true7
    for name, t in types.items():
        ok3 = t == true3
        p3, lost, won = mcnemar(base3, ok3)
        row = {'spice': spice, 'model': name, 'n_photos': len(true3), 'acc_3class': ok3.mean() * 100,
               'pure_called_adulterated': int(np.sum((true3 == 'pure') & (t != 'pure'))),
               'adulterated_called_pure': int(np.sum((true3 != 'pure') & (t == 'pure'))),
               'vs_resnet_fixed': won, 'vs_resnet_broken': lost, 'mcnemar_p_3class': p3}
        if name in full:
            ok7 = full[name] == true7
            row.update({'acc_type_and_level': ok7.mean() * 100, 'level_fixed': int(np.sum(~base7 & ok7)),
                        'level_broken': int(np.sum(base7 & ~ok7))})
        rows.append(row)
    for k, i in enumerate(te_ids):
        ref.append({'spice': spice, 'photo': os.path.basename(D['items'][i][0]), 'label': true7[k],
                    'classes3': '|'.join(order), 'pipeline_p3': '|'.join(f'{v:.5f}' for v in p_p3[k]),
                    'classes7': '|'.join(r_cls7), 'resnet_p7': '|'.join(f'{v:.5f}' for v in r_p7[k]),
                    'voted_type': types['Both voting'][k], 'final_label': full['Both voting'][k]})
    # things needed to rebuild the pipeline models on the backend machine
    gw = int(spice == 'red_chilli_powder')
    for part in ['trainval', 'test']:
        shutil.copy(os.path.join(FEAT_DIR, f'feats_{spice}_{part}_gw{gw}.npz'), os.path.join(bundle, 'features'))
    step = max(1, len(te_ids) // 4)
    for i in te_ids[::step][:4]:
        p, lab = D['items'][i]
        dst = os.path.join(bundle, 'photos', spice, lab)
        os.makedirs(dst, exist_ok=True)
        if not LOCAL:
            shutil.copy(p, dst)
    print(f'{spice}: done')

res = pd.DataFrame(rows).round(2)
res.to_csv(os.path.join(VOTE_DIR, 'vote_results.csv'), index=False)
pd.DataFrame(ref).to_csv(os.path.join(VOTE_DIR, 'vote_per_photo.csv'), index=False)
shutil.copy(os.path.join(VOTE_DIR, 'vote_per_photo.csv'), bundle)
zip_path = shutil.make_archive(os.path.join(VOTE_DIR, 'deploy_bundle'), 'zip', bundle)
print(res.to_string(index=False))
print('\nBundle:', zip_path, f'{os.path.getsize(zip_path) / 1e6:.1f} MB')
print('VOTE TEST FINISHED')